In [1]:
#!pip install azure-functions-durable==1.2.9

In [2]:
%load_ext autoreload
%autoreload 2

In [22]:
import logging
import sys
import os
sys.path.append('../')
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import azure.functions as func
import azure.durable_functions as df
import functools
from typing import Callable
from centraal_dataframework.constants import TASK_PARAM_NAME

#from centraal_dataframework.blueprints import runner
#from centraal_dataframework.resources import GreatExpectationsToolKit, get_context, get_datalake

In [6]:
STR_FMT = 'TAREA: %(name)s--%(asctime)s-%(levelname)s-%(message)s'

In [7]:
bp = df.Blueprint()

## Internal working




In [8]:
myApp = df.DFApp(http_auth_level=func.AuthLevel.ANONYMOUS)

In [31]:
import json
from dataclasses import dataclass, field
from typing import Callable, List
import azure.functions as func
import inspect
from azure.durable_functions import Blueprint, DurableOrchestrationContext
from azure.functions import FunctionApp


@dataclass
class TaskInfo:
    succeeded: bool
    args: dict = field(default_factory=dict)

    def to_json(self):
        return json.dumps(self.__dict__)

    @staticmethod
    def from_json(json_str):
        data = json.loads(json_str)
        return TaskInfo(**data)


class Task:
    def __init__(self, func: Callable):
        self.func = func
        self.name = func.__name__
    
    def _check_func(self):
        signature = inspect.signature(self.func)
        parameters = signature.parameters
        
        if TASK_PARAM_NAME not in parameters:
            raise TypeError(f"El parametro debe ser llamado {TASK_PARAM_NAME}")
        
        param = parameters[TASK_PARAM_NAME]
        param_type = param.annotation
        
        if not param_type==TaskInfo:
            raise ValueError(f"El parametro debe ser tipo `TaskInfo`")     
    
    def __call__(self):

        self._check_func()

        return self

In [10]:

def get_bindings_type(bp, key_name: str = "activity", type:str = "activityTrigger")-> List[str]:
    list_of_bindings = [f._function.get_bindings_dict()['bindings'][0] for f in bp._function_builders]
    names_of_activities = [bd[key_name] for bd in list_of_bindings if bd["type"]==type]
    return names_of_activities

def check_if_func_name_is_register(bp: Blueprint, name: str):
    if name in get_bindings_type(bp):
        if name == "alert":
            logging.debug("alert es una función del framework. No se hace log de la advertencia")
            return True
        logging.warning("Ignorando funcion con nombre duplicado %s", name)
        return True
    return False

def check_if_func_name_is_orchestator(bp: Blueprint, name: str):
    if name in get_bindings_type(bp, key_name= "orchestration", type="orchestrationTrigger"):
        logging.error("No es posible registrar pipeline %s. El nombre ya se encuentra definido", name)
        raise ValueError(f"{name} ya es un pipeline registrado. Los nombres de pipelines deben ser unicos.")
    return False

In [23]:
def alert_function(task_info: TaskInfo):
    """Funcion que envia una alerta."""
    print("alerting!")
    url_logic_app = os.environ.get("logic_app_url")
    args_error = TaskInfo.args
    name_task_failed = args_error["name_task_failed"]
    print(name_task_failed, "fallo!")

In [ ]:
class PipelineConfiguration:
    def __init__(self) -> None:
        self.url_logic_app = ""
        self.emails_notificar : List[str] = []
        

class FrameworkConfiguration:

    def __init__(self) -> None:
        pass


    def get_pipe_config(self, name: str) -> PipelineConfiguration:
        return


 def _load_conf(self) -> dict:
    archivo = os.environ.get("YAML_CONFIGURACION", "centraal_dataframework.yaml")
    logger.info("cargando configuracion desde %s", archivo)
    try:
        with open(archivo, 'r', encoding="utf-8") as file:
            data = yaml.safe_load(file)
            self.conf = data
    except FileNotFoundError as error_file:
        logger.warning("se carga archivo dummy %s", error_file, exc_info=True)
        self.conf = {"tareas": {}, "emails_notificar": [], "url_logicapp_email": "url"}

In [24]:
class Pipeline:

    def __init__(self, name: str, tasks: List[Task], bp: Blueprint, config:FrameworkConfiguration = None) -> None:

        self.tasks: List[Task] = tasks
        self.name = name
        self.route = f"start_{self.name}"
        self.config: PipelineConfiguration = config.get_pipe_config(self.name)
        self.build(bp)

    def _register_tasks(self, bp: Blueprint):
        
        for t in self.tasks:
            if t.name == "alert":
                raise ValueError("La tarea `alert` es una actividad del framework, por favor usar otro nombre.")

            task_in_bp  = check_if_func_name_is_register(bp, t.name)

            if not task_in_bp:
                bp.activity_trigger(input_name="task_info", activity=t.name)(t.func)

    def __create_orchestator_fun(self) -> Callable:
        def orchestator(context: DurableOrchestrationContext):
            results = {}

            try:
                for t in self.tasks:
                    if not results:
                        result = yield context.call_activity(t.name, TaskInfo(True, {"some": "parameter"}))

                    else:
                        result[t.name] = yield context.call_activity(t.name, result)

                    results[t.name] = result

            except Exception as e:
                yield context.call_activity(
                    "alert",
                    TaskInfo(
                        succeeded=False,
                        args={
                            "name_task_failed": t.name,
                            "task_executed": [n for n in results.keys()],
                            "error": str(e),
                            "emails": self.config.emails_notificar,
                            "url_logic": self.config.url_logic_app
                        },
                    ),
                )

                raise ValueError("Error en pipeline")

            return results

        return orchestator

    def __create_start_fun(self) -> Callable:
        async def start_orchestrator_func(req: func.HttpRequest, client):
            instance_id = await client.start_new(self.route)
            logging.info(f"Started orchestration with ID = '{instance_id}'.")
            return client.create_check_status_response(req, instance_id)

        return start_orchestrator_func

    def _create_register_orchestartor(self, bp: Blueprint):
        fun_orch = self.__create_orchestator_fun()
        check_if_func_name_is_orchestator(bp, self.name)
        bp.orchestration_trigger(context_name="context", orchestration=self.name)(fun_orch)

    def _create_register_start(self, bp: Blueprint):
        fun_start = self.__create_start_fun()
        bp.route(route=self.route)(bp.durable_client_input(client_name=f"client_{self.name}")(fun_start))

    
    def register_pipeline(self, bp: Blueprint):
        self._register_tasks(bp)
        self._create_register_orchestartor(bp)
        self._create_register_start(bp)

    def register_alert_activity(self, bp: Blueprint):

        if check_if_func_name_is_register(bp, "alert"):
            logging.debug("Ya se ha registrado la funcion de alerta")
        
        bp.activity_trigger(input_name="task_info", activity="alert")(alert_function)
            

    def build(self, bp: Blueprint):
        self.register_alert_activity(bp)
        self.register_pipeline(bp)

We want to able to schedule these pipeline, so we wil create an instance of `runner` that will contains all those pipelines

> note: Is a table a proper a better way to storage this information?


The framework will have 2 `standard` functions:
1. a function that will have a timer trigger (60 minutes), that will check with the runner which pipelines needs to be executed

> note: we assume that this function will start each pipeline and with the control of [Configuration throttles](https://learn.microsoft.com/en-us/azure/azure-functions/durable/durable-functions-perf-and-scale#configuration-of-throttles) we will able to limit the concurrency, by default the framework enable one `pipeline` and one `task` each time. The important question is: Can we start all the pipelines needed and this configuration will manage the concurrency without any intervention?

also it seems nthat we need to take into account the `partitionCount`

> Note: If the user want to trigger a specific pipeline we will recommend to start trough rest API the same method `run_pipeline`. Also **Right now we asume that when the user perform that operation, the durable function framework add to the existing queue without any intervention.

2. A function to get the status of a pipeline -> this can be useful to leverage other external monitoring acitivities. The idea is to have a http trigger.

> can we also use this to get the status of the pipeline running?

In [29]:
import os
import datetime
import requests


DUMMY_STRING = "DefaultEndpointsProtocol=https;AccountName=na;AccountKey=key!=k;EndpointSuffix=core.windows.net"
OVERWRITE_TAREA = os.environ.get("SOBREESCRIBIR_TAREA", True)
TAREA_DEBE_ESTAR_CONF = os.environ.get("TAREA_DEBE_CONFIGURACION", True)
UTC_OFFSET = datetime.timedelta(hours=os.environ.get("UTC_OFFSET", -5))


In [47]:
class Runner:
    """Clase que representa el runner."""

    def __init__(self, pipelines: List[Pipeline]) -> None:
        """Constructor."""
        self.pipelines = {}
        if "AZURE_STORAGE_CONNECTION_STRING" not in os.environ:
            logger.warning("AZURE_STORAGE_CONNECTION_STRING No configurado!")
            os.environ["AZURE_STORAGE_CONNECTION_STRING"] = DUMMY_STRING
        function_name = os.environ.get("host")
        self.url = f"https://{function_name}.azurewebsites.net/api"
        self.url_status = f"https://{function_name}.azurewebsites.net/runtime/webhooks/durabletask/instances"


    def get_pipeline(self, pipe_name: str) -> Pipeline:
        """Obtiene la tarea."""
        return self.pipelines.get(pipe_name)

    def es_programable(self, task_name_conf: dict, fecha_ejecucion: datetime = None) -> bool:
        """Verificar que la tarea puede ser programada."""
        if fecha_ejecucion is None:
            fecha_ejecucion = datetime.datetime.utcnow().replace(tzinfo=datetime.timezone.utc)

        allowed_days = str(task_name_conf["dias"])
        allowed_hours = str(task_name_conf["horas"])

        if allowed_days == "*" and allowed_hours == "*":
            return True

        fecha_en_hora_local: datetime = fecha_ejecucion + UTC_OFFSET
        allowed_days = allowed_days.split(",")
        allowed_hours = allowed_hours.split(",")
        hora_ejecucion = str(fecha_en_hora_local.time().hour)
        dia_ejecucion = str(fecha_en_hora_local.date().weekday())

        if hora_ejecucion in allowed_hours and dia_ejecucion in allowed_days:
            return True

        if allowed_days == ["*"] and hora_ejecucion in allowed_hours:
            return True

        if allowed_hours == ["*"] and dia_ejecucion in allowed_days:
            return True

        return False

    def get_tareas_programables(self, fecha_ejecucion: datetime = None) -> List[str]:
        """Obtiene las tareas que pueden se programadas."""
        tareas = []
        for task, task_conf in self.conf["tareas"].items():
            if self.es_programable(task_conf, fecha_ejecucion):
                tareas.append(task)
        return tareas


    def run_pipeline(self, pipeline_name: str):
        """Ejecuta el pipeline."""
        pipeline_to_execute = self.get_pipeline(pipeline_name)

        if pipeline_to_execute is None:
            raise ValueError("Pipeline no definido.")
        
        response = requests.post(f"{self.url}/{pipeline_to_execute.route}")
        if response.status_code == 202:
            instance_id = response.json()["id"]
            return instance_id
     
        raise ValueError("Error en inicio de pipeline")
    
    def get_status(self, pipeline_name: str):
        response = requests.get(self.url_status)
        if response.status_code == 200:
            status = response.json()["runtimeStatus"]
            return status
        else:
            print("Error checking status")
            return None
   
    
    def register_timer_triger_func(self, app: FunctionApp):
        runner = self
        def check_and_schedule_pipelines(mytimer: func.TimerRequest):
            """Verifica que tareas se deben programar.

            Args:
                mytimer: objeto TimerRequest requerido por la función.
            """
            utc_timestamp = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=datetime.timezone.utc)

            if mytimer.past_due:
                logging.warning('La función se ejecuto a fuera de tiempo')

            tareas = runner.get_tareas_programables(utc_timestamp)
            if tareas is None:
                logger.warning("No se programaron tareas")
            return "Pipeline programados"
    
        app.schedule(schedule="0 0 * * * *", arg_name="mytimer", run_on_startup=False)(check_and_schedule_pipelines)
    
    def register_get_status(self, app: FunctionApp):
        runner = self
        def get_status(req: func.HttpRequest) -> func.HttpResponse:
            """Adiciona una tarea.

            Args:
                req: request de la funcion, debe tener el parametro llamado "task_name",
                para ejecutar una lista de funciones en especifico.
            """
            try:
                req_body = req.get_json()
                task_names = req_body.get('pipeline', [])

            except ValueError:
                return func.HttpResponse(
                    "La funcion tuvo un error",
                    status_code=400,
                )

            if task_names is None:
                return func.HttpResponse("No se enviaron tareas", status_code=400)
                # se ejecuta tarea con el nombre especifico sin importar el archivo de configuracion
            runner.get_status(task_names)
            return func.HttpResponse(f"status de tareas: {task_names}", status_code=200)

        app.route(methods=['post'])(get_status)
    
    
    def register_framework_functions(self, app: FunctionApp):
        self.register_timer_triger_func(app)
        self.register_get_status(app)

In [48]:
bp = df.Blueprint()

In [49]:
@Task
def first_step_of_pipeline(task_info: TaskInfo):
    print("first step")


@Task
def second_step_of_pipeline(task_info: TaskInfo):
    print("second step")


@Task
def other_step_of_pipeline(task_info: TaskInfo):
    print("second step")

pipeline = Pipeline(name="test_pipeline", tasks=[first_step_of_pipeline, second_step_of_pipeline], bp=bp)

otherpipeline = Pipeline(name="other_pipeline", tasks=[first_step_of_pipeline, other_step_of_pipeline, second_step_of_pipeline], bp=bp)

pipeline_test = Pipeline(name="test_pipeline_2", tasks=[first_step_of_pipeline], bp=bp)

In [50]:
app = func.FunctionApp(http_auth_level=func.AuthLevel.FUNCTION)

runner = Runner(pipelines = [pipeline, pipeline_test])

app.register_functions(bp)
runner.register_framework_functions(app)


In [51]:
fs = app.get_functions()

for f in fs:
    print(f)

{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "activityTrigger", "name": "task_info", "activity": "alert"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "activityTrigger", "name": "task_info", "activity": "first_step_of_pipeline"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "activityTrigger", "name": "task_info", "activity": "second_step_of_pipeline"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "orchestrationTrigger", "name": "context", "orchestration": "test_pipeline"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "durableClient", "name": "client_test_pipeline"}, {"direction": "IN", "type": "httpTrigger", "name": "req", "authLevel": "FUNCTION", "route": "start_test_pipeline"}, {"direction": "OUT", "type": "http", "name": "$return"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "activityTrigger", "nam